In [ ]:
# IMPORTING LIBRARIES

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

from tensorflow.keras.utils import to_categorical

In [ ]:
df = pd.read_csv("/content/netflix_titles.csv")

df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [ ]:
# KEEP ONLY NECESSARY COLUMNS

df = df[['description', 'type']]

# REMOVE MISSING VALUES
df.dropna(inplace=True)

df.head()

                                         description     type
0  As her father nears the end of his life, filmm...    Movie
1  After crossing paths at a party, a Cape Town t...  TV Show
2  To protect his family from a powerful drug lor...  TV Show
3  Feuds, flirtations and toilet talk go down amo...  TV Show
4  In a city of coaching centers known to train I...  TV Show


In [ ]:
# CONVERT LABELS INTO NUMBERS

encoder = LabelEncoder()

df['type'] = encoder.fit_transform(df['type'])

# Movie = 0
# TV Show = 1

print(df['type'].value_counts())

type
0    6131
1    2676
Name: count, dtype: int64


In [ ]:
# INPUT AND OUTPUT

X = df['description']
y = df['type']

In [ ]:
# CONVERT TEXT INTO NUMBERS

tokenizer = Tokenizer(num_words=5000)

tokenizer.fit_on_texts(X)

X_sequences = tokenizer.texts_to_sequences(X)

In [ ]:
# MAKE ALL SEQUENCES SAME LENGTH

max_length = 100

X_padded = pad_sequences(X_sequences, maxlen=max_length)

In [ ]:
# SPLIT DATA

X_train, X_test, y_train, y_test = train_test_split(
    X_padded,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(7045, 100)
(1762, 100)


In [ ]:
# BUILD SIMPLE RNN MODEL

model = Sequential()

# EMBEDDING LAYER
model.add(Embedding(input_dim=5000, output_dim=64, input_length=max_length))

# SIMPLE RNN LAYER
model.add(SimpleRNN(64))

# OUTPUT LAYER
model.add(Dense(1, activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
# COMPILE MODEL

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy'])

In [ ]:
# SHOW MODEL PARAMETERS

model.build(input_shape=(None, max_length))

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 100, 64)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 328,321 (1.25 MB)

 Trainable params: 328,321 (1.25 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# TRAIN MODEL

history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_test, y_test))

Epoch 1/5
221/221 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - accuracy: 0.6975 - loss: 0.6114 - val_accuracy: 0.6918 - val_loss: 0.6093
Epoch 2/5
221/221 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - accuracy: 0.7906 - loss: 0.4469 - val_accuracy: 0.6538 - val_loss: 0.6586
Epoch 3/5
221/221 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - accuracy: 0.9693 - loss: 0.1032 - val_accuracy: 0.6856 - val_loss: 0.8549
Epoch 4/5
221/221 ━━━━━━━━━━━━━━━━━━━━ 6s 21ms/step - accuracy: 0.9993 - loss: 0.0132 - val_accuracy: 0.6481 - val_loss: 1.1255
Epoch 5/5
221/221 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - accuracy: 0.9994 - loss: 0.0047 - val_accuracy: 0.6532 - val_loss: 1.2012


In [ ]:
# TEST ACCURACY

loss, accuracy = model.evaluate(X_test, y_test)

print("Accuracy:", accuracy)

56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6532 - loss: 1.2012
Accuracy: 0.6532349586486816


In [ ]:
sample_texts = ["A group of teenagers fight supernatural monsters",
                "A romantic movie based on true events",
                "New episodes released every week",
                "A documentary film about wildlife",
                "Crime investigation continues in season two"]

In [ ]:
# MAKE PREDICTIONS

# Convert sample texts to sequences
sample_sequences = tokenizer.texts_to_sequences(sample_texts)

# Pad sample sequences
sample_pad = pad_sequences(sample_sequences, maxlen=max_length)

predictions = model.predict(sample_pad)

for text, pred in zip(sample_texts, predictions):

    label = encoder.inverse_transform(np.array([1 if pred > 0.5 else 0]))[0] # Use encoder to get original labels

    print("Text:", text)
    print("Prediction:", label)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step
Text: A group of teenagers fight supernatural monsters
Prediction: Movie
Text: A romantic movie based on true events
Prediction: Movie
Text: New episodes released every week
Prediction: Movie
Text: A documentary film about wildlife
Prediction: Movie
Text: Crime investigation continues in season two
Prediction: Movie


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Bidirectional

# BiRNN Model
birnn_model = Sequential()

birnn_model.add(
    Embedding(
        input_dim=5000,
        output_dim=64))

birnn_model.add(
    Bidirectional(
        SimpleRNN(64)))

birnn_model.add(Dense(32, activation='relu'))

birnn_model.add(Dense(1, activation='sigmoid'))

# Build model before summary
birnn_model.build(input_shape=(None, max_length))

birnn_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 100, 64)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 340,673 (1.30 MB)

 Trainable params: 340,673 (1.30 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
birnn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy'])

In [ ]:
history_birnn = birnn_model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_test, y_test))

Epoch 1/5
221/221 ━━━━━━━━━━━━━━━━━━━━ 9s 30ms/step - accuracy: 0.7016 - loss: 0.5951 - val_accuracy: 0.7435 - val_loss: 0.5471
Epoch 2/5
221/221 ━━━━━━━━━━━━━━━━━━━━ 6s 28ms/step - accuracy: 0.8525 - loss: 0.3358 - val_accuracy: 0.7174 - val_loss: 0.6585
Epoch 3/5
221/221 ━━━━━━━━━━━━━━━━━━━━ 7s 32ms/step - accuracy: 0.9841 - loss: 0.0520 - val_accuracy: 0.7191 - val_loss: 0.9983
Epoch 4/5
221/221 ━━━━━━━━━━━━━━━━━━━━ 9s 27ms/step - accuracy: 0.9993 - loss: 0.0062 - val_accuracy: 0.7128 - val_loss: 1.1744
Epoch 5/5
221/221 ━━━━━━━━━━━━━━━━━━━━ 7s 30ms/step - accuracy: 0.9996 - loss: 0.0041 - val_accuracy: 0.7145 - val_loss: 1.0530


In [ ]:
loss, accuracy = birnn_model.evaluate(X_test, y_test)

print("BiRNN Accuracy:", accuracy)

56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.7145 - loss: 1.0530
BiRNN Accuracy: 0.7145289182662964


In [ ]:
sample_texts = [
    "A group of friends embark on a dangerous mission.",
    "A documentary about wildlife and nature.",
    "An exciting TV series full of mystery.",
    "A romantic movie set in Paris.",
    "A crime thriller with unexpected twists."]

sample_seq = tokenizer.texts_to_sequences(sample_texts)
sample_pad = pad_sequences(sample_seq, maxlen=max_length)

predictions = birnn_model.predict(sample_pad)

for text, pred in zip(sample_texts, predictions):
    label = "TV Show" if pred > 0.5 else "Movie"
    print(f"Text: {text}")
    print(f"Prediction: {label}")
    print("-" * 50)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 182ms/step
Text: A group of friends embark on a dangerous mission.
Prediction: Movie
--------------------------------------------------
Text: A documentary about wildlife and nature.
Prediction: Movie
--------------------------------------------------
Text: An exciting TV series full of mystery.
Prediction: TV Show
--------------------------------------------------
Text: A romantic movie set in Paris.
Prediction: Movie
--------------------------------------------------
Text: A crime thriller with unexpected twists.
Prediction: Movie
--------------------------------------------------
